# Train DQN on WasteSegregationEnv

This notebook is fully self-contained: run every cell top to bottom and it will clone the repository, install dependencies, train all ten DQN hyperparameter presets, and save models/logs back into the repository structure.

See `README.md` in the repository root for the full project writeup, environment design, and the six-mechanic reward specification.

## 1. Clone the repository and install dependencies

In [ ]:
!git clone https://github.com/kelvintawe12/Summative-Reinforcement-Learning.git
%cd Summative-Reinforcement-Learning
!pip install -q -e .
!pip install -q tensorboard nbformat

## 2. Imports

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from environment.custom_env import WasteSegregationEnv
from training.dqn_training import train_single_run
import os, pandas as pd

## 3. Hyperparameter presets

The ten DQN configurations required for the hyperparameter-sweep deliverable, generated from `training/hyperparameters.py` so this notebook stays in sync with the canonical source of truth. Each preset is fully versioned here for reproducibility.

In [ ]:
DQN_PRESETS = [
    {
        "run_id": "dqn_01",
        "total_timesteps": 60000,
        "learning_rate": 0.001,
        "buffer_size": 20000,
        "batch_size": 32,
        "gamma": 0.99,
        "exploration_fraction": 0.3,
        "exploration_final_eps": 0.05,
        "target_update_interval": 500,
        "train_freq": 4,
        "learning_starts": 1000
    },
    {
        "run_id": "dqn_02",
        "total_timesteps": 60000,
        "learning_rate": 0.0005,
        "buffer_size": 20000,
        "batch_size": 32,
        "gamma": 0.99,
        "exploration_fraction": 0.3,
        "exploration_final_eps": 0.05,
        "target_update_interval": 500,
        "train_freq": 4,
        "learning_starts": 1000
    },
    {
        "run_id": "dqn_03",
        "total_timesteps": 60000,
        "learning_rate": 0.00025,
        "buffer_size": 20000,
        "batch_size": 64,
        "gamma": 0.99,
        "exploration_fraction": 0.3,
        "exploration_final_eps": 0.05,
        "target_update_interval": 500,
        "train_freq": 4,
        "learning_starts": 1000
    },
    {
        "run_id": "dqn_04",
        "total_timesteps": 60000,
        "learning_rate": 0.001,
        "buffer_size": 50000,
        "batch_size": 64,
        "gamma": 0.99,
        "exploration_fraction": 0.3,
        "exploration_final_eps": 0.05,
        "target_update_interval": 500,
        "train_freq": 4,
        "learning_starts": 1000
    },
    {
        "run_id": "dqn_05",
        "total_timesteps": 60000,
        "learning_rate": 0.001,
        "buffer_size": 20000,
        "batch_size": 32,
        "gamma": 0.95,
        "exploration_fraction": 0.3,
        "exploration_final_eps": 0.05,
        "target_update_interval": 500,
        "train_freq": 4,
        "learning_starts": 1000
    },
    {
        "run_id": "dqn_06",
        "total_timesteps": 60000,
        "learning_rate": 0.001,
        "buffer_size": 20000,
        "batch_size": 32,
        "gamma": 0.999,
        "exploration_fraction": 0.3,
        "exploration_final_eps": 0.05,
        "target_update_interval": 500,
        "train_freq": 4,
        "learning_starts": 1000
    },
    {
        "run_id": "dqn_07",
        "total_timesteps": 60000,
        "learning_rate": 0.001,
        "buffer_size": 20000,
        "batch_size": 32,
        "gamma": 0.99,
        "exploration_fraction": 0.1,
        "exploration_final_eps": 0.02,
        "target_update_interval": 500,
        "train_freq": 4,
        "learning_starts": 1000
    },
    {
        "run_id": "dqn_08",
        "total_timesteps": 60000,
        "learning_rate": 0.001,
        "buffer_size": 20000,
        "batch_size": 32,
        "gamma": 0.99,
        "exploration_fraction": 0.5,
        "exploration_final_eps": 0.1,
        "target_update_interval": 500,
        "train_freq": 4,
        "learning_starts": 1000
    },
    {
        "run_id": "dqn_09",
        "total_timesteps": 60000,
        "learning_rate": 0.001,
        "buffer_size": 20000,
        "batch_size": 32,
        "gamma": 0.99,
        "exploration_fraction": 0.3,
        "exploration_final_eps": 0.05,
        "target_update_interval": 100,
        "train_freq": 4,
        "learning_starts": 1000
    },
    {
        "run_id": "dqn_10",
        "total_timesteps": 60000,
        "learning_rate": 0.001,
        "buffer_size": 20000,
        "batch_size": 32,
        "gamma": 0.99,
        "exploration_fraction": 0.3,
        "exploration_final_eps": 0.05,
        "target_update_interval": 2000,
        "train_freq": 4,
        "learning_starts": 1000
    }
]

## 4. Train all ten runs

Trains each preset in sequence, saving a model checkpoint to `models/dqn/<run_id>.zip` and a per-episode reward CSV log to `logs/dqn/<run_id>.csv`.

In [ ]:
log_dir = "logs/dqn"
model_dir = "models/dqn"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

saved_paths = []
for preset in DQN_PRESETS:
    print(f"Training {preset['run_id']} ...")
    path = train_single_run(preset, log_dir, model_dir, seed=0)
    saved_paths.append(path)
    print(f"  saved -> {path}")

print("\nAll DQN runs complete.")

## 5. Visualize training

A 4-panel dashboard across all ten DQN runs (raw reward, smoothed reward, the DQN diagnostic curve, and episode length), followed by the hyperparameter + performance table with the best run highlighted. All plotting lives in `results/notebook_viz.py` so it stays consistent across notebooks.

In [ ]:
from results.notebook_viz import training_dashboard, show_hyperparameter_table

training_dashboard('dqn', save_to='results/plots/dqn_dashboard.png')
table = show_hyperparameter_table('dqn')
table

## 6. Persist results

Colab sessions are ephemeral, so models and logs must be persisted somewhere durable. Two options are provided below -- use whichever fits your workflow. **Only run one of the two cells.**

In [ ]:
# Option A: mount Google Drive and copy results there
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/tawe_kelvin_rl_summative_results
!cp -r {log_dir} /content/drive/MyDrive/tawe_kelvin_rl_summative_results/
!cp -r {model_dir} /content/drive/MyDrive/tawe_kelvin_rl_summative_results/
print("Copied logs/ and models/ to Google Drive.")

In [ ]:
# Option B: commit and push results directly back to the GitHub repository
# Requires a GitHub personal access token with repo write access.
GITHUB_TOKEN = ""  # paste your token here (do not commit this notebook with a real token)
if GITHUB_TOKEN:
    !git add {log_dir} {model_dir}
    !git commit -m "Add DQN training results"
    !git push https://{GITHUB_TOKEN}@github.com/kelvintawe12/Summative-Reinforcement-Learning.git HEAD:main
else:
    print("Skipped: no GitHub token provided.")

## 7. Quick evaluation

Loads the best of the ten runs (by mean reward over the last 10% of episodes) and runs one evaluation episode as a fast sanity check before moving to the full analysis pipeline locally (`python -m results.analysis`).

In [ ]:
best_run_id, best_mean = None, -float('inf')
for preset in DQN_PRESETS:
    csv_path = os.path.join(log_dir, preset['run_id'] + '.csv')
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    tail = df.tail(max(1, len(df) // 10))
    mean_r = tail['episode_reward'].mean()
    if mean_r > best_mean:
        best_mean, best_run_id = mean_r, preset['run_id']

print(f'Best run: {best_run_id} (mean reward last 10%: {best_mean:.2f})')

best_model_path = os.path.join(model_dir, best_run_id + '.zip')
model = DQN.load(best_model_path)

eval_env = WasteSegregationEnv(seed=999)
obs, info = eval_env.reset(seed=999)
done, total_reward = False, 0.0
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(int(action))
    total_reward += reward
    done = terminated or truncated

print(f'Evaluation episode reward: {total_reward:.2f}')
print('Episode stats:', info['episode_stats'])

## 8. Compile everything into a downloadable zip

Regenerates the full analysis (hyperparameter tables + every plot: reward curves, DQN objective / policy-entropy curves, convergence subplots, best-run comparison) and packages `logs/`, `models/`, and `results/` into a single archive. In Colab this triggers a browser download so you can drop the results straight into the local repo (and into the report) after the session ends.

In [ ]:
from results.notebook_viz import bundle_results

bundle_results(out_path='dqn_results_bundle.zip')